In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("yellow-nov-2025")
    .getOrCreate()
)

spark.version

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/02 12:03:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


'3.5.1'

In [4]:
df = spark.read.parquet("yellow_tripdata_2025-11.parquet")

df.printSchema()
df.show(5)
df.count()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)

+--------+--------------------+---------------------+---------------+------

4181444

In [5]:
df = df.repartition(4)
df.write.parquet("yellow_nov_2025_repartitioned")

In [6]:
import os
os.listdir("yellow_nov_2025_repartitioned")

['part-00000-4517216d-3863-404f-9987-a7cb2ea221e0-c000.snappy.parquet',
 '.part-00001-4517216d-3863-404f-9987-a7cb2ea221e0-c000.snappy.parquet.crc',
 '.part-00003-4517216d-3863-404f-9987-a7cb2ea221e0-c000.snappy.parquet.crc',
 'part-00003-4517216d-3863-404f-9987-a7cb2ea221e0-c000.snappy.parquet',
 '_SUCCESS',
 'part-00001-4517216d-3863-404f-9987-a7cb2ea221e0-c000.snappy.parquet',
 '.part-00002-4517216d-3863-404f-9987-a7cb2ea221e0-c000.snappy.parquet.crc',
 'part-00002-4517216d-3863-404f-9987-a7cb2ea221e0-c000.snappy.parquet',
 '._SUCCESS.crc',
 '.part-00000-4517216d-3863-404f-9987-a7cb2ea221e0-c000.snappy.parquet.crc']

In [7]:
import os

folder = "yellow_nov_2025_repartitioned"

files = [
    os.path.join(folder, f)
    for f in os.listdir(folder)
    if f.endswith(".parquet")   # only real parquet files
]

sizes_mb = [os.path.getsize(f) / (1024 * 1024) for f in files]

print("Number of parquet files:", len(files))
print("Individual sizes (MB):", [round(s,2) for s in sizes_mb])
print("Average size (MB):", round(sum(sizes_mb)/len(sizes_mb), 2))

Number of parquet files: 4
Individual sizes (MB): [24.39, 24.4, 24.41, 24.41]
Average size (MB): 24.4


In [11]:
from pyspark.sql.functions import to_date, col

df_nov15 = df.filter(
    to_date(col("tpep_pickup_datetime")) == "2025-11-15"
)

df_nov15.count()

162604

In [12]:
from pyspark.sql.functions import col, unix_timestamp, max as spark_max

df_with_duration = df.withColumn(
    "trip_hours",
    (unix_timestamp(col("tpep_dropoff_datetime")) -
     unix_timestamp(col("tpep_pickup_datetime"))) / 3600
)

df_with_duration.select(
    spark_max("trip_hours")
).show()

+-----------------+
|  max(trip_hours)|
+-----------------+
|90.64666666666666|
+-----------------+



In [13]:
zones = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("taxi_zone_lookup.csv")
)

zones.show(5)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows



In [14]:
from pyspark.sql.functions import count

pickup_counts = (
    df.groupBy("PULocationID")
      .agg(count("*").alias("trip_count"))
)

result = (
    pickup_counts
    .join(zones, pickup_counts.PULocationID == zones.LocationID)
    .select("Zone", "trip_count")
    .orderBy("trip_count")
)

result.show(10)

+--------------------+----------+
|                Zone|trip_count|
+--------------------+----------+
|Governor's Island...|         1|
|       Arden Heights|         1|
|Eltingville/Annad...|         1|
|       Port Richmond|         3|
|   Rossville/Woodrow|         4|
|       Rikers Island|         4|
| Green-Wood Cemetery|         4|
|         Great Kills|         4|
|         Jamaica Bay|         5|
|         Westerleigh|        12|
+--------------------+----------+
only showing top 10 rows

